# Устойчивость и конкурирующие объяснения

Насколько результат зависит от выбора окон, способа снятия рыночной
компоненты и состава выборки. И проверка альтернативного объяснения —
возврата к среднему после роста.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import moex, pipeline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
P = moex.DATA_PROCESSED

## Пятнадцать спецификаций

In [2]:
robustness = pd.read_parquet(P / 'robustness.parquet')
for event_type in ['включение', 'исключение']:
    part = robustness[robustness.event_type == event_type]
    print(f'--- {event_type} ---')
    print(f'  оценок: {len(part)}, диапазон от {part.оценка.min():+.1%} до {part.оценка.max():+.1%}')
    print(f'  отрицательных: {(part.оценка < 0).sum()} из {len(part)}')
    print(f'  значимых после поправки FDR: {(part.p_fdr < 0.05).sum()}')
    print()
robustness[['event_type', 'измерение', 'вариант', 'n', 'оценка', 'медиана',
            'p_value', 'p_fdr']].round(4)

--- включение ---
  оценок: 15, диапазон от -19.7% до -6.4%
  отрицательных: 15 из 15
  значимых после поправки FDR: 12

--- исключение ---
  оценок: 15, диапазон от -7.4% до +9.5%
  отрицательных: 4 из 15
  значимых после поправки FDR: 0



,event_type,измерение,вариант,n,оценка,медиана,p_value,p_fdr
0,включение,окно события,"[+1,+20]",14,-0.0876,-0.0775,0.0719,0.1348
1,исключение,окно события,"[+1,+20]",12,0.0340,0.0374,0.1178,0.1766
2,включение,окно события,"[+1,+40]",14,-0.1441,-0.1492,0.0066,0.0248
3,исключение,окно события,"[+1,+40]",12,0.0563,0.0718,0.0661,0.1323
4,включение,окно события,"[+1,+60]",14,-0.1830,-0.2421,0.0028,0.0141
5,исключение,окно события,"[+1,+60]",12,0.0387,-0.0053,0.2383,0.2749
6,включение,окно события,"[+1,+90]",14,-0.1973,-0.2238,0.0013,0.0141
7,исключение,окно события,"[+1,+90]",12,0.0946,0.0707,0.0411,0.0949
8,включение,окно оценки,"[-250,-40]",14,-0.1830,-0.2421,0.0028,0.0141
9,исключение,окно оценки,"[-250,-40]",12,0.0387,-0.0053,0.2383,0.2749


## Эффект отбора

Снижение цены после включения без предшествующего роста плохо ложится на
механику индексного эффекта. Конкурирующее объяснение: в индекс попадают
бумаги, которые уже выросли, а дальше работает возврат к среднему.

In [3]:
data = pipeline.load()
index_prices = data.index_prices.sort_values('TRADEDATE')
calendar = data.calendar
market = pd.Series(np.log(index_prices.CLOSE.values[1:] / index_prices.CLOSE.values[:-1]),
                   index=calendar[1:])

def pre_event_excess(ticker, event_date, window=(-250, -31)):
    frame = data.quotes[(data.quotes.SECID == ticker) & data.quotes.CLOSE.notna()].sort_values('TRADEDATE')
    position = calendar.searchsorted(event_date)
    dates = calendar[max(0, position + window[0]):max(0, position + window[1])]
    part = frame[frame.TRADEDATE.isin(dates)]
    if len(part) < 100:
        return np.nan
    return np.log(part.CLOSE.iloc[-1] / part.CLOSE.iloc[0]) - market[market.index.isin(dates)].sum()

rows = []
for event in data.price_sample.itertuples():
    rows.append({'ticker': event.ticker, 'event_type': event.event_type,
                 'pre': pre_event_excess(event.ticker, event.event_date)})
pre = pd.DataFrame(rows).dropna()

for event_type, part in pre.groupby('event_type'):
    print(f'{event_type}: среднее {part.pre.mean():+.1%}, медиана {part.pre.median():+.1%}, '
          f'положительных {(part.pre > 0).sum()} из {len(part)}')

включение: среднее +19.0%, медиана +16.4%, положительных 13 из 14
исключение: среднее -40.5%, медиана -24.2%, положительных 1 из 12


Биржа включает то, что выросло, и исключает то, что упало — прямое
следствие правил отбора по капитализации и ликвидности.

In [4]:
from src import abnormal_returns as arm
from scipy import stats

abnormal = pd.read_parquet(P / 'abnormal_returns.parquet')
car = arm.car_by_event(abnormal[abnormal.event_type == 'включение'], (1, 60))
merged = car.merge(pre[pre.event_type == 'включение'], on='ticker')

correlation = stats.spearmanr(merged.pre, merged.car)
print(f'корреляция Спирмена между доходностью до и после: {correlation.statistic:+.3f} '
      f'(p = {correlation.pvalue:.3f}, n = {len(merged)})')

корреляция Спирмена между доходностью до и после: -0.574 (p = 0.032, n = 14)


## Решающая проверка

Если падение целиком объясняется отбором, то контроль, выросший так же
сильно, должен падать так же сильно, и разница обнулится.

In [5]:
from src import matching, inference

for flag, name in ((False, 'размер и ликвидность'), (True, 'плюс динамика до события')):
    pairs = matching.match_controls(
        data.price_sample, data.quotes, data.securities, data.sectors,
        data.composition, data.events, data.calendar, data.control_pool,
        match_on_momentum=flag, index_prices=data.index_prices)
    returns = arm.matched_returns(pairs, data.quotes, data.calendar, (1, 60))
    print(f'--- матчинг: {name} ---')
    for event_type, part in returns.groupby('event_type'):
        part = part.dropna(subset=['difference'])
        test = inference.clustered_test(part.difference, part.event_date)
        print(f'  {event_type:11s} {part.difference.mean():+7.1%}  p = {test["p_value"]:.3f}')
    print(matching.covariate_balance(pairs).round(3).to_string(index=False))
    print()

--- матчинг: размер и ликвидность ---
  включение     -6.4%  p = 0.083
  исключение    -6.7%  p = 0.291
               признак  среднее в тесте  среднее в контроле  стандартизованная разность
логарифм капитализации           25.607              25.232                       0.406
 логарифм числа сделок            9.582               9.217                       0.511



--- матчинг: плюс динамика до события ---
  включение    -17.3%  p = 0.013
  исключение    +2.9%  p = 0.126
               признак  среднее в тесте  среднее в контроле  стандартизованная разность
логарифм капитализации           25.607              25.283                       0.410
 логарифм числа сделок            9.582               8.676                       1.488
 доходность до события           -0.089              -0.061                      -0.072



Разница не обнуляется, а увеличивается. Возврат к среднему объясняет часть
падения, но не всё.

Оговорка: добавление третьего признака ухудшает баланс по ликвидности, так
что две спецификации не вполне сопоставимы. Полностью развести механизмы
выборка из 14 включений не позволяет.